# Phase 2: Indexing, Slicing, Filtering & Stacking
> **Target Audience:** Beginners to Intermediate  
> **Prerequisites:** Phase 1 (Array creation and shapes)  
> **Key Goal:** Extract, filter, manipulate, and combine multi-dimensional data with zero-copy efficiency.

Welcome to **Phase 2** of the NumPy Guide! In data science and machine learning, most of your time is spent querying and reshaping data subsets.

### In this notebook, you will learn:
1. **1D Indexing & Slicing** – Positive indices, negative indices, strides, and reversal.
2. **2D & Multi-dimensional Indexing** – Rows, columns, and submatrix blocks.
3. **The View vs. Copy Trap** – Understanding memory views vs independent copies.
4. **Sorting & Ordering** – `np.sort()` across axes and `np.argsort()` for ranking.
5. **Boolean Masking & Filtering** – Filtering with condition masks (`&`, `|`, `~`).
6. **Fancy Indexing & `np.where()`** – Index arrays and ternary conditional labeling.
7. **Joining & Resizing Data** – `np.concatenate()`, `np.vstack()`, `np.hstack()`, and `np.delete()`.
8. **Real-World Case Study** – Retail store quarterly revenue analytics.

## 1. 1D Indexing & Slicing
NumPy uses standard Python slice syntax `array[start:stop:step]` with extended support for negative strides:

In [1]:
import numpy as np

arr = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])

print("Full array          :", arr)
print("Basic slice [2:7]   :", arr[2:7])      # Elements at index 2, 3, 4, 5, 6
print("With step [1:8:2]   :", arr[1:8:2])    # Every 2nd element
print("Negative index [-3] :", arr[-3])       # 3rd element from the end
print("Reversed [::-1]     :", arr[::-1])     # Array in reverse order

Full array          : [ 10  20  30  40  50  60  70  80  90 100]
Basic slice [2:7]   : [30 40 50 60 70]
With step [1:8:2]   : [20 40 60 80]
Negative index [-3] : 80
Reversed [::-1]     : [100  90  80  70  60  50  40  30  20  10]


## 2. 2D & Multi-dimensional Indexing
For 2D arrays, specify indices for each dimension separated by a comma: `arr[row, col]`:
- `arr[i, j]` selects a specific scalar.
- `arr[i, :]` selects an entire row.
- `arr[:, j]` selects an entire column.
- `arr[r1:r2, c1:c2]` extracts a submatrix block.

In [2]:
grid = np.array([
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12]
])
print("2D Grid (3x4):\n", grid)

print("\nElement at row 1, col 2      :", grid[1, 2])
print("Entire row 1                  :", grid[1, :])
print("Entire column 2               :", grid[:, 2])
print("Submatrix (rows 0-1, cols 1-2):\n", grid[0:2, 1:3])

2D Grid (3x4):
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

Element at row 1, col 2      : 7
Entire row 1                  : [5 6 7 8]
Entire column 2               : [ 3  7 11]
Submatrix (rows 0-1, cols 1-2):
 [[2 3]
 [6 7]]


## 3. The View vs. Copy Trap (Critical!)
> **CRITICAL WARNING:** Slicing an array creates a **view**, NOT a copy!
> Modifying a slice directly alters the original array.
> Always call `.copy()` if you need an independent clone.

In [3]:
source_arr = np.array([10, 20, 30, 40, 50])

# Slice view:
slice_view = source_arr[1:4]
slice_view[0] = 9999

print("Source array after mutating slice view:")
print(source_arr)  # [10, 9999, 30, 40, 50] -> MUTATED!

# Safe approach using .copy():
source_arr2 = np.array([10, 20, 30, 40, 50])
safe_copy = source_arr2[1:4].copy()
safe_copy[0] = 9999

print("\nSource array after mutating safe_copy:")
print(source_arr2)  # Untouched!

Source array after mutating slice view:
[  10 9999   30   40   50]

Source array after mutating safe_copy:
[10 20 30 40 50]


## 4. Sorting & Ordering
- **`np.sort(arr)`**: Returns a sorted copy of the array.
- In 2D arrays: specify `axis=0` (sort down columns) or `axis=1` (sort across rows).
- **`np.argsort(arr)`**: Returns the indices that would sort the array (essential for data ranking).

In [4]:
unsorted_1d = np.array([30, 10, 40, 20, 50])
print("Sorted 1D:", np.sort(unsorted_1d))

# argsort gives the index positions of elements from smallest to largest:
sorted_indices = np.argsort(unsorted_1d)
print("Indices that sort array:", sorted_indices)
print("Original ordered by indices:", unsorted_1d[sorted_indices])

# 2D sorting across axes:
matrix_unsorted = np.array([
    [9, 2, 7],
    [4, 8, 1]
])
print("\nSort across rows (axis=1):\n", np.sort(matrix_unsorted, axis=1))
print("Sort down columns (axis=0):\n", np.sort(matrix_unsorted, axis=0))

Sorted 1D: [10 20 30 40 50]
Indices that sort array: [1 3 0 2 4]
Original ordered by indices: [10 20 30 40 50]

Sort across rows (axis=1):
 [[2 7 9]
 [1 4 8]]
Sort down columns (axis=0):
 [[4 2 1]
 [9 8 7]]


## 5. Boolean Masking & Conditional Filtering
You can pass boolean conditions directly into the indexing brackets.

In [5]:
scores = np.array([45, 88, 72, 91, 55, 63, 79, 99])

# Boolean mask:
pass_mask = scores >= 70
print("Boolean Mask :", pass_mask)
print("Passing Scores:", scores[pass_mask])

# Combining multiple conditions using bitwise operators:
# & = AND, | = OR, ~ = NOT (Parentheses are mandatory!)
distinction = scores[(scores >= 80) & (scores <= 95)]
print("Distinction Scores (80 to 95):", distinction)

Boolean Mask : [False  True  True  True False False  True  True]
Passing Scores: [88 72 91 79 99]
Distinction Scores (80 to 95): [88 91]


## 6. Fancy Indexing & `np.where()`
- **Fancy Indexing:** Pass an array or list of integer indices to extract arbitrary elements.
- **`np.where(condition)`**: Returns the coordinates where the condition is True.
- **`np.where(condition, if_true, if_false)`**: Vectorized ternary operator.

In [6]:
data = np.array([10, 25, 40, 55, 70, 85])

# Fancy indexing: extract indices 0, 2, 4
print("Elements at [0, 2, 4]:", data[[0, 2, 4]])

# Finding indices where condition is met:
idx = np.where(data > 50)
print("Indices where data > 50:", idx[0])

# Conditional labeling (ternary):
status = np.where(data >= 50, "Pass", "Fail")
print("Status Labels:", status)

Elements at [0, 2, 4]: [10 40 70]
Indices where data > 50: [3 4 5]
Status Labels: ['Fail' 'Fail' 'Fail' 'Pass' 'Pass' 'Pass']


## 7. Joining, Stacking & Deleting Data
- `np.concatenate((a, b), axis=0)`: Join along existing dimension.
- `np.vstack((a, b))` & `np.hstack((a, b))`: Convenience functions for vertical and horizontal stacking.
- `np.delete(arr, indices, axis)`: Delete elements along an axis.

In [7]:
row_a = np.array([[1, 2, 3]])
row_b = np.array([[4, 5, 6]])

# Vertical stack (adds rows)
stacked_v = np.vstack((row_a, row_b))
print("Vertical Stack (vstack):\n", stacked_v)

# Horizontal stack (adds columns)
col_c = np.array([[7], [8]])
stacked_h = np.hstack((stacked_v, col_c))
print("\nHorizontal Stack (hstack):\n", stacked_h)

# Delete column index 1:
deleted_col = np.delete(stacked_h, 1, axis=1)
print("\nAfter deleting column 1:\n", deleted_col)

Vertical Stack (vstack):
 [[1 2 3]
 [4 5 6]]

Horizontal Stack (hstack):
 [[1 2 3 7]
 [4 5 6 8]]

After deleting column 1:
 [[1 3 7]
 [4 6 8]]


## 8. Real-World Case Study: Retail Store Performance Analysis
Let's apply indexing, filtering, and sorting to a simulated quarterly sales dataset:
- 10 stores across 4 quarters (Q1, Q2, Q3, Q4)
- Revenue measured in thousands of dollars ($k)

In [8]:
np.random.seed(42)
# 10 stores, 4 quarters of revenue
sales_quarterly = np.random.randint(30, 150, size=(10, 4))
print("Store Quarterly Sales ($k) [10 stores x 4 quarters]:\n", sales_quarterly)

# 1. Filter: Find stores with Q4 revenue >= 100
q4_mask = sales_quarterly[:, 3] >= 100
top_q4_stores = sales_quarterly[q4_mask]
print(f"\nStores with Q4 >= $100k: {top_q4_stores.shape[0]} stores")

# 2. Performance categorization: Classify each quarter as 'High' (>= 100) or 'Standard' (< 100)
quarter_labels = np.where(sales_quarterly >= 100, "High", "Standard")
print("Store #1 Quarter Labels:", quarter_labels[0])

# 3. Store Ranking: Calculate annual total revenue per store and rank top 3 stores
annual_totals = sales_quarterly.sum(axis=1)
# argsort sorts ascending; reverse [::-1] for descending
rank_indices = np.argsort(annual_totals)[::-1]

print("\n=== TOP 3 STORES BY ANNUAL REVENUE ===")
for rank, idx in enumerate(rank_indices[:3], start=1):
    print(f"Rank {rank}: Store #{idx} -> Total Revenue: ${annual_totals[idx]}k (Quarters: {sales_quarterly[idx]})")

Store Quarterly Sales ($k) [10 stores x 4 quarters]:
 [[132  81 122  44]
 [136 101  90  50]
 [132 112 116 104]
 [104 117 146 129]
 [133  53  32  51]
 [ 82  31 117 137]
 [ 59  67  31  93]
 [ 89  50  62 105]
 [ 87  51 137 118]
 [ 78 120  88  71]]

Stores with Q4 >= $100k: 5 stores
Store #1 Quarter Labels: ['High' 'Standard' 'High' 'Standard']

=== TOP 3 STORES BY ANNUAL REVENUE ===
Rank 1: Store #3 -> Total Revenue: $496k (Quarters: [104 117 146 129])
Rank 2: Store #2 -> Total Revenue: $464k (Quarters: [132 112 116 104])
Rank 3: Store #8 -> Total Revenue: $393k (Quarters: [ 87  51 137 118])


## 9. Hands-On Practice Challenge
**Task:**
1. Generate an array of 20 random integers between 1 and 100 (`seed=42`).
2. Replace all values greater than 50 with the value `50` (clipping / ceiling operation) using `np.where()`.
3. Filter all even numbers from the clipped array.

*Try coding your solution before checking below!*

In [9]:
# Solution:
np.random.seed(42)
raw_nums = np.random.randint(1, 101, size=20)
print("Original Numbers:\n", raw_nums)

# Clip values > 50 to 50
clipped = np.where(raw_nums > 50, 50, raw_nums)
print("\nClipped Numbers (<= 50):\n", clipped)

# Filter even numbers
evens = clipped[clipped % 2 == 0]
print("\nEven numbers from clipped array:\n", evens)

Original Numbers:
 [ 52  93  15  72  61  21  83  87  75  75  88 100  24   3  22  53   2  88
  30  38]

Clipped Numbers (<= 50):
 [50 50 15 50 50 21 50 50 50 50 50 50 24  3 22 50  2 50 30 38]

Even numbers from clipped array:
 [50 50 50 50 50 50 50 50 50 50 24 22 50  2 50 30 38]


---
### Phase 2 Summary
- [x] Slicing `[start:stop:step]` returns views; use `.copy()` for independent copies.
- [x] Multi-dimensional access: `arr[row, col]`, `arr[:, col]`, `arr[row, :]`.
- [x] Sort arrays with `np.sort()`, and rank with `np.argsort()`.
- [x] Filter with boolean masks `arr[condition]` and combine with `&`, `|`, `~`.
- [x] Conditional replacement with `np.where(cond, val_if_true, val_if_false)`.
- [x] Stack with `np.vstack()` and `np.hstack()`.

**Next Up:** Proceed to `phase-3.ipynb` to learn broadcasting rules, statistical aggregations, and business analytics!